In [ ]:
%%writefile vector_add.cu

#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// CUDA Kernel
__global__ void addVectors(int* A, int* B, int* C, int n) {

    int index = blockIdx.x * blockDim.x + threadIdx.x;

    if (index < n) {
        C[index] = A[index] + B[index];
    }
}

int main() {

    int n = 1000000;
    int size = n * sizeof(int);

    int *A, *B, *C;

    // Allocate memory on Host
    cudaMallocHost(&A, size);
    cudaMallocHost(&B, size);
    cudaMallocHost(&C, size);

    // Initialize vectors
    for (int i = 0; i < n; i++) {
        A[i] = i;
        B[i] = i * 2;
    }

    // Allocate memory on Device
    int *dev_A, *dev_B, *dev_C;

    cudaMalloc(&dev_A, size);
    cudaMalloc(&dev_B, size);
    cudaMalloc(&dev_C, size);

    // Copy Host -> Device
    cudaMemcpy(dev_A, A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, size, cudaMemcpyHostToDevice);

    // Kernel configuration
    int blockSize = 256;
    int numBlocks = (n + blockSize - 1) / blockSize;

    // Launch kernel
    addVectors<<<numBlocks, blockSize>>>(dev_A, dev_B, dev_C, n);

    // Copy Device -> Host
    cudaMemcpy(C, dev_C, size, cudaMemcpyDeviceToHost);

    // Print first 10 results
    cout << "Result (first 10 elements):\n";

    for (int i = 0; i < 10; i++) {
        cout << C[i] << " ";
    }

    cout << endl;

    // Free device memory
    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    // Free host memory
    cudaFreeHost(A);
    cudaFreeHost(B);
    cudaFreeHost(C);

    return 0;
}

Writing vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!nvidia-smi

Mon May 11 13:53:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!./vector_add

Result (first 10 elements):
0 3 6 9 12 15 18 21 24 27 
